# Download OPP-115 dataset and map annotations to good/neutral/bad

In [ ]:
# Install the Hugging Face library
!pip install datasets -q

import pandas as pd
from datasets import load_dataset

In [ ]:
# 1. Fetching dataset directly from Hugging Face
hf_dataset = load_dataset("alzoubi36/opp_115", split="train")

# Convert it into a Pandas table so we can work with it
df = pd.DataFrame(hf_dataset)

In [ ]:
# 2. Mapping numerical labels to Risk Categories
# Based on the hugging face OPP-115 schema, here is what those numbers mean:
# We map these numbers directly to our 3 risk levels:
hf_mapping = {
    0: 'neutral',  # Data Retention
    1: 'good',     # Data Security
    2: 'neutral',  # Do Not Track
    3: 'bad',      # First Party Collection/Use
    4: 'neutral',  # International and Specific Audiences
    5: 'neutral',  # Other / Introductory
    6: 'neutral',  # Policy Change
    7: 'neutral',  # Practice not covered
    8: 'neutral',  # Privacy contact information
    9: 'bad',      # Third Party Sharing/Collection
    10: 'good',    # User Access, Edit and Deletion
    11: 'good'     # User Choice/Control
}

In [ ]:
# 3. Create a logic function to handle the lists
def determine_highest_risk(label_list):
    # Convert the list of numbers into a list of our risk words
    risks = [hf_mapping.get(num, 'neutral') for num in label_list]

    # Prioritize 'bad' -> then 'good' -> default to 'neutral'
    if 'bad' in risks:
        return 'bad'
    elif 'good' in risks:
        return 'good'
    else:
        return 'neutral'

# Apply our custom function to every row
df['label'] = df['label'].apply(determine_highest_risk)

# Standardize columns
df['original_label_list'] = hf_dataset['label'] # Keep the original list just in case
df['clause_text'] = df['text']
df['dataset'] = 'OPP-115'

In [ ]:
# 4. Clean up the table
final_df = df[['clause_text', 'original_label_list', 'label', 'dataset']]
final_df = final_df.dropna(subset=['label'])

In [ ]:
# 5. Export the final file
final_df.to_csv('opp115_mapped_hf.csv', index=False)

print("\n--- Pipeline Execution Successful ---")
print(f"Total clauses processed: {len(final_df)}")
print("\nClass Breakdown:")
print(final_df['label'].value_counts())

In [ ]:
from google.colab import files

files.download('opp115_mapped_hf.csv')